# Training-free token pruning for ViT-Base, part 4: LoRA control run

**Runtime: T4 GPU, fresh runtime** (loads its own clean pretrained model, so it is not contaminated by
part 3's training).

Same LoRA recipe as part 3 (r=8 on blocks 7–11, 3 epochs, lr 1e-3), but trained with **no pruning**
(K = 0), then evaluated with pruning on (K = 50%). Comparing this model with part 3's model under the
same pruning isolates whether training *with* pruning taught the network anything pruning-specific.

> Outputs are from the recorded run; download and progress logs were stripped.

In [ ]:
!pip uninstall -y torchao -q  # Colab ships a torchao too old for peft 0.21; we do not use torchao here
!pip install -q timm peft

In [1]:
import torch, torch.nn as nn, timm, os, time, copy, tarfile, urllib.request, pandas as pd
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
assert torch.cuda.is_available(), "Switch the runtime to a GPU"
device = "cuda"; print(torch.cuda.get_device_name(0))
model = timm.create_model("vit_base_patch16_224", pretrained=True).eval().to(device)
tfm = timm.data.create_transform(**timm.data.resolve_data_config({}, model=model))

PRUNE_LAYER, TARGET_K = 6, 0.5     # TARGET_K = the pruning level we test generalization to (Day 6/8's operating point)
LORA_R, LR, EPOCHS, BATCH = 8, 1e-3, 3, 32
IMAGENETTE_WNIDS = {"n01440764":0,"n02102040":217,"n02979186":482,"n03000684":491,"n03028079":497,
 "n03394916":566,"n03417042":569,"n03425413":571,"n03445777":574,"n03888257":701}

Tesla T4


In [ ]:
class PrunedViT(nn.Module):
    """Runs blocks 0..prune_layer (inclusive), scores patches by CLS->patch attention captured
    inside block `prune_layer`, drops the lowest-scoring `drop_frac` of patches, then runs the
    remaining blocks on the shorter sequence. Call .close() to restore the model."""
    def __init__(self, model, prune_layer=6, drop_frac=0.5):
        super().__init__()
        self.model, self.prune_layer, self.drop_frac = model, prune_layer, drop_frac
        attn = model.blocks[prune_layer].attn
        self._attn_mod, self._orig_fused = attn, attn.fused_attn
        attn.fused_attn = False
        self._attn = None
        self._h = attn.attn_drop.register_forward_hook(self._grab)
    def _grab(self, mod, inp, out):
        self._attn = inp[0]
    def close(self):
        self._h.remove(); self._attn_mod.fused_attn = self._orig_fused
    def forward(self, x):
        m = self.model
        x = m.patch_embed(x); x = m._pos_embed(x); x = m.patch_drop(x); x = m.norm_pre(x)
        for blk in m.blocks[: self.prune_layer + 1]:
            x = blk(x)
        B, N, D = x.shape
        n_patches = N - 1
        n_drop = int(n_patches * self.drop_frac)
        if n_drop > 0:
            scores = self._attn.float().mean(dim=1)[:, 0, 1:]
            kept = scores.topk(n_patches - n_drop, dim=1, largest=True).indices.sort(dim=1).values
            patches = x[:, 1:].gather(1, kept.unsqueeze(-1).expand(-1, -1, D))
            x = torch.cat([x[:, :1], patches], dim=1)
        for blk in m.blocks[self.prune_layer + 1 :]:
            x = blk(x)
        x = m.norm(x)
        return m.forward_head(x)

## Data

In [2]:
if not os.path.exists("imagenette2-320"):
    urllib.request.urlretrieve("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz", "imagenette.tgz")
    tarfile.open("imagenette.tgz").extractall(".")

tfm_train = timm.data.create_transform(**timm.data.resolve_data_config({}, model=model), is_training=True)
train_ds = ImageFolder("imagenette2-320/train", transform=tfm_train)
to_in1k_train = torch.tensor([IMAGENETTE_WNIDS[w] for w in train_ds.classes])
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
print(len(train_ds), "training images,", len(train_loader), "steps/epoch")

val_ds = ImageFolder("imagenette2-320/val", transform=tfm)
to_in1k_val = torch.tensor([IMAGENETTE_WNIDS[w] for w in val_ds.classes])
xs_, ys_ = [], []
for xb, yb in DataLoader(val_ds, batch_size=64, num_workers=2):
    xs_.append(xb.half()); ys_.append(yb)
Xval = torch.cat(xs_); yval_in1k = to_in1k_val[torch.cat(ys_)]
print(Xval.shape, "val images cached")

@torch.no_grad()
def eval_acc(fn, drop_frac, bs=64):
    """Full-precision (fp32) eval on the cached val set at a given drop_frac - matches Day 4/6/8 methodology."""
    was_training, was_K = fn.training, fn.drop_frac
    fn.eval(); fn.drop_frac = drop_frac
    logits = torch.cat([fn(Xval[i:i+bs].to(device).float()).float().cpu() for i in range(0, len(Xval), bs)])
    fn.train(was_training); fn.drop_frac = was_K
    return (logits.argmax(1) == yval_in1k).float().mean().item()

9469 training images, 295 steps/epoch
torch.Size([3925, 3, 224, 224]) val images cached


## Attach LoRA (targets blocks 7-11, same recipe as before) and set training K=0

In [3]:
from peft import LoraConfig, get_peft_model

# CONTROL RUN: train with K=0 (no pruning ever seen during training) — the opposite of the
# earlier K=0.5 run. Same recipe otherwise (r, lr, epochs, target modules) so the two runs
# are comparable. This isolates whether LoRA specifically adapts blocks 7-11 to tolerate a
# shortened sequence, vs. just making the closed 10-class task trivially easy either way.
TRAIN_K = 0.0
target_modules = [f"blocks.{L}.{sub}" for L in range(PRUNE_LAYER + 1, 12)
                  for sub in ("attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2")]
lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_R * 2, lora_dropout=0.05, bias="none",
                      target_modules=target_modules)
peft_model = get_peft_model(model, lora_cfg)
peft_model.print_trainable_parameters()

pruner = PrunedViT(model, prune_layer=PRUNE_LAYER, drop_frac=TRAIN_K)

trainable params: 491,520 || all params: 87,059,176 || trainable%: 0.5646


## Train with K=0 (no pruning ever seen)

In [4]:
import torch.nn.functional as F
baseline_acc = eval_acc(pruner, TRAIN_K)
print(f"Before training: acc @ block {PRUNE_LAYER} / K={TRAIN_K:.0%} = {baseline_acc:.4f}  (Day 6 frozen result: 0.895)")

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scaler = torch.amp.GradScaler("cuda")
pruner.train()
for epoch in range(EPOCHS):
    t0, running = time.time(), 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), to_in1k_train[yb].to(device)
        opt.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = F.cross_entropy(pruner(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        running += loss.item()
    val_acc = eval_acc(pruner, TRAIN_K)
    print(f"epoch {epoch+1}/{EPOCHS}: train loss {running/len(train_loader):.4f}, val acc @ K={TRAIN_K:.0%} = {val_acc:.4f}  ({time.time()-t0:.0f}s)")

Before training: acc @ block 6 / K=0% = 0.8945  (Day 6 frozen result: 0.895)
epoch 1/3: train loss 0.0840, val acc @ K=0% = 0.9977  (111s)
epoch 2/3: train loss 0.0602, val acc @ K=0% = 0.9967  (110s)
epoch 3/3: train loss 0.0637, val acc @ K=0% = 0.9975  (111s)


## Merge, then evaluate at both K=0 and K=50% (the pruning level it never trained on)

In [5]:
pruner.close()
peft_model.eval()
merged = peft_model.merge_and_unload()
pruner_final = PrunedViT(merged, prune_layer=PRUNE_LAYER, drop_frac=TRAIN_K)

acc_at_K0   = eval_acc(pruner_final, 0.0)          # matches its own training condition
acc_at_K0p5 = eval_acc(pruner_final, TARGET_K)     # generalization to pruning it NEVER saw during training

print(f"K=0-trained model, eval @ K=0:                {acc_at_K0:.4f}")
print(f"K=0-trained model, eval @ K={TARGET_K:.0%} (unseen at train time): {acc_at_K0p5:.4f}")

K=0-trained model, eval @ K=0:                0.9975
K=0-trained model, eval @ K=50% (unseen at train time): 0.9962


## Full comparison

In [6]:
# Fill in the two numbers you already have from the earlier (K=0.5-trained) run.
K0p5_trained_at_K0p5 = 0.9959   # <- from that run's "After LoRA + merge" print
K0p5_trained_at_K0   = 0.9962   # <- from that run's post-hoc drop_frac=0.0 check

print(f"{'condition':45s}{'acc':>8s}")
print(f"{'-'*53}")
print(f"{'frozen, no LoRA, K=0 (Day 6)':45s}{0.895:>8.4f}")
print(f"{'frozen, no LoRA, K='+f'{TARGET_K:.0%}'+' (Day 6)':45s}{0.886:>8.4f}")
print(f"{'K='+f'{TARGET_K:.0%}'+'-trained, eval @ K='+f'{TARGET_K:.0%}':45s}{K0p5_trained_at_K0p5:>8.4f}")
print(f"{'K='+f'{TARGET_K:.0%}'+'-trained, eval @ K=0':45s}{K0p5_trained_at_K0:>8.4f}")
print(f"{'K=0-trained, eval @ K=0 (this run)':45s}{acc_at_K0:>8.4f}")
print(f"{'K=0-trained, eval @ K='+f'{TARGET_K:.0%}'+' (this run)':45s}{acc_at_K0p5:>8.4f}")
print()
gap = (acc_at_K0 - acc_at_K0p5) * 100
print(f"K=0-trained model's drop when pruned at inference (never seen during training): {gap:.2f} points")
print("If this gap is small (~0-1 pt): the task is just easy after any fine-tuning; pruning barely matters regardless of training regime.")
print("If this gap is large (several points): fine-tuning WITH pruning specifically taught tolerance to it — a real LoRA-for-pruning effect.")

condition                                         acc
-----------------------------------------------------
frozen, no LoRA, K=0 (Day 6)                   0.8950
frozen, no LoRA, K=50% (Day 6)                 0.8860
K=50%-trained, eval @ K=50%                    0.9959
K=50%-trained, eval @ K=0                      0.9962
K=0-trained, eval @ K=0 (this run)             0.9975
K=0-trained, eval @ K=50% (this run)           0.9962
K=0-trained model's drop when pruned at inference (never seen during training): 0.13 points
If this gap is small (~0-1 pt): the task is just easy after any fine-tuning; pruning barely matters regardless of training regime.
If this gap is large (several points): fine-tuning WITH pruning specifically taught tolerance to it — a real LoRA-for-pruning effect.


## Result
Under 50% pruning at evaluation, the model trained **without** pruning scores **99.62%**, and part 3's
model trained "with" pruning scores **99.59%**: a −0.03-point difference, i.e. **no measurable
pruning-specific benefit** from LoRA on this task. This control model loses only 0.13 points when
pruning is switched on at inference, so after any fine-tuning the 10-class task is easy enough that
pruning barely matters.

Note the "before training" accuracy here (0.8945, unpruned) equals what part 3 printed for its
supposedly pruned model before training, and part 3's epochs were not faster than these. Both suggest
pruning was not active during part 3's training (see the open issue in part 3), so this comparison
cannot rule out a small effect; it only shows none was observed.